In [ ]:
# configure pyspark session
#.appName allows you to name the session
#.master allows you too allocate number of cores ie local[*] grants use of all cores
#.config allows you to allocate RAM. Rule of thumb is to use 1/2 of what you have on your pc
#.getorcreate() start the session

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('plumber') \
    .master ('local[*]') \
    .config ("spark.driver.memory", "16g") \
    .getOrCreate()

## Bronze Layer

In [ ]:
# Libraries and variables

import json
import pandas as pd
from pathlib import Path

datapath = Path('../data') #Path to directory containing data
bronze_playlist_path = Path ('../bronze/playlist')
bronze_playlist_path.mkdir(exist_ok = True)

bronze_sliceinfo_path = Path ('../bronze/sliceinfo')
bronze_sliceinfo_path.mkdir(exist_ok = True)

In [ ]:
# writing 'playlist' into ../bronze/
for file in datapath.iterdir():
    with open (file, 'r') as f:
        jread = json.load(f)
    df = pd.json_normalize (jread['playlists'])

    out_name = file.stem.replace('mpd.slice.', 'slices_') + '.parquet'
    df.to_parquet(bronze_playlist_path / out_name)

In [ ]:
# writing 'sliceinfo' into ../bronze/
for file in datapath.iterdir():
    with open (file,'r') as f:
        jread=json.load(f)
    df = pd.json_normalize([jread['info']]) #trying to normalize to the info level

    out_name = file.stem.replace('mpd.slice.', 'slices_') + '.parquet'
    df.to_parquet(bronze_sliceinfo_path / out_name)

## Silver Layer

In [ ]:
# Define a function that converts empyt strings "", " ", "   "...etc into properl nulls
def emptystringconv (df, columns = None):
    """
    This function will read string columns in a dataframe and convert any whitespace string values to proper null type.
    If no columns are passed, function will loop through all string columns
    """
    if columns is None:
        for x, y in df.dtypes:
            if y == "string":
                df = df.withColumn(x, F.when(F.trim(F.col(x)) == "", F.lit(None)).otherwise(F.col(x)))

    else:
        for x in columns:
            df = df.withColumn(x, F.when(F.col(x) == "", F.lit(None)).otherwise(F.col(x)))

    return df


In [ ]:
#Read in bronze playlist table
dfp = spark.read.parquet("../bronze/playlist") 

In [ ]:
#playlist table
playlist_t = emptystringconv(dfp)
playlist_t = playlist_t.select("pid", \
                        "name", \
                        "collaborative", \
                        "modified_at", \
                        "num_tracks", \
                        "num_albums", \
                        "num_followers", \
                        "num_edits",\
                        "num_artists", \
                        "duration_ms", \
                        "description") \
                        .withColumn("collaborative", F.col("collaborative").cast("boolean"))

playlist_t.printSchema()

In [ ]:
#track table
track_t = flat.select("track_uri",\
                "track_name", \
                "artist_uri", \
                "duration_ms") \
                .dropDuplicates(["track_uri"])

track_t.printSchema()

In [ ]:
#artist table 
artist_t = flat.select ("artist_uri", \
                "artist_name") \
                .dropDuplicates(["artist_uri"])

artist_t.printSchema()

In [ ]:
#album table
album_t = flat.select("album_uri", \
                "album_name") \
                .dropDuplicates(['album_uri'])

album_t.printSchema()

In [ ]:
#pos_bridge table
pos_bridge_t = flat.select(
    "pid", \
    "pos", \
    "track_uri"
).distinct()

pos_bridge_t.count()

In [ ]:
#write
playlist_t.write.parquet("../silver/playlist")
track_t.write.parquet("../silver/track")
artist_t.write.parquet("../silver/artist")
album_t.write.parquet("../silver/album")
pos_bridge_t.write.parquet("../silver/pid_pos")